[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cca-cce/osm-comm/blob/main/jnb/skod52/lecture_lab_3.ipynb)

---
title: "Applications of NLP"
subtitle: "SKOD52 · Module 3, Lecture-Lab 3 · companion notebook"
format:
  html:
    toc: true
    code-overflow: wrap
jupyter: python3
execute:
  warning: false
---

This notebook continues from Lecture-Lab 2. It reads the DataFrames that lab 2
saved as `.tsv` files and applies five kinds of analysis to them:

1. **Word frequencies**: the top 10 tokens in each text
2. **Lexical analysis**: keyword in context (KWIC), named entities (NER) and
   parts of speech (POS), read through stakeholder and legitimacy theory
3. **Dictionary sentiment**: AFINN scores per token, aggregated per text
4. **Sentence sentiment**: TextBlob polarity and subjectivity per sentence,
   averaged per text
5. **Document-term matrix**: word frequencies and TF-IDF across the corpus

The whole class first runs the notebook on the **Equinor example**. Then each
project group swaps in its own lab 2 files and edits the cells marked
**▶ Your project**. Every result is saved as a `.tsv` table and a `.png`
figure in `text-data/lab3/`, and is overwritten on each run.

## 0 · Setup

On Google Colab we install the AFINN sentiment lexicon and TextBlob. Locally,
this step is skipped.

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "afinn", "textblob"], check=True)

print("Running on Colab:", IN_COLAB)

In [ ]:
import math, re, string, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from afinn import Afinn
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

pd.set_option("display.max_colwidth", 80)

ORANGE, GRAY = "#f05a22", "#9e9e9e"
POS_COLOR, NEG_COLOR = "#2a9d8f", "#d1495b"      # positive / negative sentiment

## 1 · Read the lab 2 output

### The data folder

As in lab 2, the notebook works with a folder called **`text-data`** next to
the notebook (`/content/text-data` on Colab). All lab 3 results go into the
subfolder `text-data/lab3/`.

> **▶ Your project:** set `ORG` to a short name for your organization, in
> lowercase with no spaces (for example `"shell"`). It is used in the
> output file names.

In [ ]:
ORG = "equinor"     # ← EDIT: short name of your organization

DATA_DIR = Path("text-data")
OUT_DIR = DATA_DIR / "lab3"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data folder:  ", DATA_DIR.resolve())
print("Output folder:", OUT_DIR.resolve())

### Copy the lab 2 files from Google Drive

**Before the lab:** upload the `.tsv` files that lab 2 produced
(`df_texts.tsv`, `df_sentences.tsv`, `df_tokens.tsv`) to the same Google
Drive folder as your PDF and `.txt` files.

Then edit `DRIVE_DIR` below so it points to that folder, and run the cell.
Colab asks for permission to mount your Drive. The `.tsv` files are copied
into the local `text-data/` folder; nothing is written to your Drive.
Locally, this cell is skipped.

In [ ]:
LAB2_FILES = ["df_texts.tsv", "df_sentences.tsv", "df_tokens.tsv"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    # ← EDIT this path to point to your folder of PDF, .txt and .tsv files
    DRIVE_DIR = Path("/content/drive/MyDrive/doc/comm/skod52/2025/org3-equinor-pdf")

    for name in LAB2_FILES:
        if (DRIVE_DIR / name).exists():
            shutil.copy2(DRIVE_DIR / name, DATA_DIR / name)     # overwrite local copy
            print("copied", name)
        else:
            print(f"WARNING: {name} not found in {DRIVE_DIR}")

### Read the DataFrames

`keep_default_na=False` keeps empty cells as empty strings. Without it,
pandas would turn the empty `token_ner` cells (tokens that are not part of a
named entity) into `NaN`.

In [ ]:
def read_tsv(name):
    return pd.read_csv(DATA_DIR / name, sep="\t", keep_default_na=False)


df_texts = read_tsv("df_texts.tsv")
df_sentences = read_tsv("df_sentences.tsv")
df_tokens = read_tsv("df_tokens.tsv")

# the tokens lab 2 kept for analysis: alphabetic, no stop words, no OOV, 3+ letters
kept = df_tokens[df_tokens["token_keep"]]

TEXT_IDS = df_texts["text_id"].tolist()

def short(text_id, n=28):
    """Shorten long text ids for plot labels (drops Drive's "copy-of-" prefix)."""
    text_id = re.sub(r"^copy-of-", "", text_id)
    return text_id if len(text_id) <= n else text_id[:n - 1] + "…"

print(f"{len(df_texts)} texts, {len(df_sentences)} sentences, "
      f"{len(df_tokens)} tokens ({len(kept)} kept)")
df_texts[["text_id", "text_wordcount"]]

## 2 · Top 10 tokens per text

We count the kept **lemmas** in each text and plot the 10 most frequent ones,
with one small bar chart (a *facet*) per text.

In [ ]:
df_top10 = (kept.groupby(["text_id", "token_lemma"]).size()
                .rename("n").reset_index()
                .sort_values(["text_id", "n", "token_lemma"], ascending=[True, False, True])
                .groupby("text_id").head(10))
df_top10.to_csv(OUT_DIR / f"freq-top10-{ORG}.tsv", sep="\t", index=False)   # overwrite
df_top10.head(10)

In [ ]:
ncols = 3
nrows = max(1, math.ceil(len(TEXT_IDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3 * nrows), squeeze=False)
axes = axes.ravel()

for ax, text_id in zip(axes, TEXT_IDS):
    top = df_top10[df_top10["text_id"] == text_id]
    ax.barh(top["token_lemma"][::-1], top["n"][::-1], color=ORANGE)
    ax.set_title(short(text_id, 36), fontsize=9)
    ax.tick_params(labelsize=8)

for ax in axes[len(TEXT_IDS):]:      # hide unused grid cells
    ax.axis("off")

fig.suptitle(f"Top 10 tokens per text ({ORG})")
fig.tight_layout()
fig.savefig(OUT_DIR / f"freq-top10-{ORG}.png", dpi=150)   # overwrite
plt.show()

## 3 · Lexical analysis: stakeholders and legitimacy

Lecture-Lab 1 introduced theories that help us make sense of what
organizations say about sustainability:

| Theory | Key question | What to look for in the text |
|---|---|---|
| Stakeholder theory (Freeman; Mitchell et al.) | Whom does the organization address, and who counts? | stakeholder nouns, named organizations, places and groups |
| Legitimacy theory (Suchman) | How does it justify what it does? | evaluative words such as *responsible*, *safe*, *transparent* |
| Aspirational talk (Christensen et al.) | Does it describe what it *will* do or what it *has* done? | modal verbs (*will*, *should*) vs past-tense verbs |

We work **abductively**. First we look at patterns in the data (frequent
words, entities, verb forms). When a pattern surprises us, we ask which theory
could explain it, and then go back to the text with KWIC to check whether that
explanation holds.

> **▶ Your project:** run this section on the Equinor example with the whole
> class first. Then edit `KEYWORDS` and `STAKEHOLDER_TERMS` below to fit your
> own organization, and re-run the section.

In [ ]:
# keywords for KWIC: a keyword matches every word that STARTS with it,
# so "communit" finds both "community" and "communities"
KEYWORDS = ["stakeholder", "communit", "employee", "transparen", "responsib"]   # ← EDIT
WINDOW = 7    # words of context on each side

# nouns (lemmas) that name stakeholder groups
STAKEHOLDER_TERMS = {                                                          # ← EDIT
    "stakeholder", "community", "employee", "worker", "people", "customer",
    "investor", "shareholder", "owner", "supplier", "partner", "contractor",
    "government", "authority", "regulator", "society", "union", "ngo",
    "indigenous", "neighbour", "public", "user",
}

### 3.1 Keyword in context (KWIC)

KWIC shows every occurrence of a keyword in the **raw text** with the words
around it. It is the quickest way to see *how* a word is used, not just how
often.

In [ ]:
PUNCT = string.punctuation + "“”‘’«»–—"

def kwic(keyword, window=WINDOW):
    """Every occurrence of keyword (as a word prefix) with left and right context."""
    rows = []
    for text_id, text in zip(df_texts["text_id"], df_texts["text_raw"]):
        words = text.split()
        for i, word in enumerate(words):
            if word.lower().strip(PUNCT).startswith(keyword):
                rows.append({"text_id": text_id,
                             "left": " ".join(words[max(0, i - window):i]),
                             "keyword": word,
                             "right": " ".join(words[i + 1:i + 1 + window])})
    return pd.DataFrame(rows, columns=["text_id", "left", "keyword", "right"])


kwic_tables = {}
for kw in KEYWORDS:
    kwic_tables[kw] = kwic(kw)
    kwic_tables[kw].to_csv(OUT_DIR / f"kwic-{kw}.tsv", sep="\t", index=False)   # overwrite
    print(f"{kw:<12} {len(kwic_tables[kw]):>4} hits  →  kwic-{kw}.tsv")

Read the lines for one keyword. Change the keyword to read another one.

In [ ]:
kwic_tables[KEYWORDS[0]][["left", "keyword", "right"]].head(15)

How often does each keyword occur in each text?

In [ ]:
df_kwic_counts = (pd.concat([t.assign(kw=kw) for kw, t in kwic_tables.items()])
                    .groupby(["text_id", "kw"]).size()
                    .unstack(fill_value=0)
                    .reindex(index=TEXT_IDS, columns=KEYWORDS, fill_value=0))
df_kwic_counts.to_csv(OUT_DIR / f"kwic-{ORG}.tsv", sep="\t")   # overwrite

ax = df_kwic_counts.rename(index=short)[::-1].plot.barh(
    stacked=True, figsize=(8, 0.45 * len(TEXT_IDS) + 1.5), colormap="tab10", width=0.75)
ax.set_xlabel("Keyword hits")
ax.set_ylabel("")
ax.set_title(f"KWIC keyword hits per text ({ORG})")
ax.legend(title="keyword", fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / f"kwic-{ORG}.png", dpi=150)   # overwrite
plt.show()

### 3.2 Named entities (NER): who is mentioned?

spaCy tagged each token with an entity type in lab 2 (`token_ner`). Tokens
next to each other with the same type form one entity, for example *United*
+ *Nations*. We keep the types that point to possible stakeholders.

In [ ]:
ENT_TYPES = {"ORG": "organization", "GPE": "country/city", "NORP": "group/nationality",
             "PERSON": "person", "LAW": "law/standard"}

rows, current = [], None
for tok in df_tokens.itertuples(index=False):
    continues = (current is not None and tok.token_ner == current["type"]
                 and tok.text_id == current["text_id"]
                 and tok.sentence_id == current["sentence_id"])
    if continues:
        current["words"].append(tok.token_text)
        continue
    if current is not None:
        rows.append(current)
        current = None
    if tok.token_ner:
        current = {"text_id": tok.text_id, "sentence_id": tok.sentence_id,
                   "type": tok.token_ner, "words": [tok.token_text]}
if current is not None:
    rows.append(current)

df_ents = pd.DataFrame(rows)
df_ents["entity"] = df_ents["words"].str.join(" ")
df_ents = df_ents[df_ents["type"].isin(ENT_TYPES)]

df_ner = (df_ents.groupby(["entity", "type"])
                 .agg(n=("text_id", "size"), n_texts=("text_id", "nunique"))
                 .reset_index()
                 .sort_values(["n", "entity"], ascending=[False, True]))
df_ner.to_csv(OUT_DIR / f"ner-{ORG}.tsv", sep="\t", index=False)   # overwrite
df_ner.head(15)

In [ ]:
top_ents = df_ner.head(15)[::-1]
type_colors = dict(zip(ENT_TYPES, ["#f05a22", "#264653", "#2a9d8f", "#e9c46a", "#8d6e63"]))

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_ents["entity"], top_ents["n"], color=top_ents["type"].map(type_colors))
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in type_colors.values()]
ax.legend(handles, [f"{t} ({d})" for t, d in ENT_TYPES.items()], fontsize=8, loc="lower right")
ax.set_xlabel("Mentions")
ax.set_title(f"Most mentioned named entities ({ORG})")
fig.tight_layout()
fig.savefig(OUT_DIR / f"ner-{ORG}.png", dpi=150)   # overwrite
plt.show()

### 3.3 Parts of speech (POS): how does the organization talk?

Nouns show **what and whom** the texts are about. Adjectives show how the
organization **evaluates** itself and its actions, which is where legitimacy
claims live. Verbs show what it **does**. Nouns from `STAKEHOLDER_TERMS` are
highlighted in orange.

In [ ]:
pos_rows = []
for pos in ["NOUN", "ADJ", "VERB"]:
    counts = kept.loc[kept["token_pos"] == pos, "token_lemma"].value_counts().head(10)
    for lemma, n in counts.items():
        pos_rows.append({"pos": pos, "lemma": lemma, "n": n,
                         "stakeholder": lemma in STAKEHOLDER_TERMS})

stake = (kept.loc[kept["token_pos"].eq("NOUN") & kept["token_lemma"].isin(STAKEHOLDER_TERMS),
                  "token_lemma"].value_counts().head(10))
for lemma, n in stake.items():
    pos_rows.append({"pos": "STAKEHOLDER NOUN", "lemma": lemma, "n": n, "stakeholder": True})

df_pos = pd.DataFrame(pos_rows)
df_pos.to_csv(OUT_DIR / f"pos-{ORG}.tsv", sep="\t", index=False)   # overwrite
df_pos.groupby("pos").head(3)

In [ ]:
panels = [("NOUN", "Top nouns"), ("STAKEHOLDER NOUN", "Stakeholder nouns"),
          ("ADJ", "Top adjectives"), ("VERB", "Top verbs")]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, (pos, title) in zip(axes.ravel(), panels):
    d = df_pos[df_pos["pos"] == pos][::-1]
    ax.barh(d["lemma"], d["n"], color=np.where(d["stakeholder"], ORANGE, GRAY))
    ax.set_title(title, fontsize=10)
    ax.tick_params(labelsize=8)
fig.suptitle(f"Parts of speech ({ORG}); orange = stakeholder term")
fig.tight_layout()
fig.savefig(OUT_DIR / f"pos-{ORG}.png", dpi=150)   # overwrite
plt.show()

### 3.4 Aspirational talk: will vs did

Christensen et al. call statements about intentions and ambitions
*aspirational talk*. A simple indicator: count **modal verbs** (tag `MD`:
*will, would, can, should, must, may*) and **past-tense verbs** (tag `VBD`)
per 100 verbs in each text. Here we use all tokens, because lab 2 removed
*will* and *can* as stop words.

In [ ]:
verbs = df_tokens[df_tokens["token_pos"].isin(["VERB", "AUX"])]

df_modal = (verbs.groupby("text_id")
                 .agg(verbs=("token_tag", "size"),
                      modal=("token_tag", lambda t: (t == "MD").sum()),
                      past=("token_tag", lambda t: (t == "VBD").sum()))
                 .reindex(TEXT_IDS))
df_modal["modal_per_100"] = (100 * df_modal["modal"] / df_modal["verbs"]).round(1)
df_modal["past_per_100"] = (100 * df_modal["past"] / df_modal["verbs"]).round(1)
df_modal.to_csv(OUT_DIR / f"pos-modality-{ORG}.tsv", sep="\t")   # overwrite
df_modal

In [ ]:
ax = (df_modal[["modal_per_100", "past_per_100"]]
        .rename(index=short, columns={"modal_per_100": "modal (will, should …)",
                                      "past_per_100": "past tense"})[::-1]
        .plot.barh(figsize=(8, 0.45 * len(TEXT_IDS) + 1.5), color=[ORANGE, GRAY], width=0.75))
ax.set_xlabel("Per 100 verbs")
ax.set_ylabel("")
ax.set_title(f"Aspirational (modal) vs performance (past) talk ({ORG})")
plt.tight_layout()
plt.savefig(OUT_DIR / f"pos-modality-{ORG}.png", dpi=150)   # overwrite
plt.show()

### Sensemaking

Discuss in your group, and write down your answers for the project:

1. **Stakeholders:** which groups and named actors appear most, and which ones
   you would expect are missing? Check a few with KWIC: are they *addressed*,
   *described*, or only *listed*?
2. **Legitimacy:** do the top adjectives point to pragmatic (useful, valuable),
   moral (responsible, fair, transparent) or cognitive (standard, recognized)
   legitimacy?
3. **Aspirational talk:** which texts lean on *will* and *should*, and which
   report what has been done? Does that fit the topic of the page?
4. **Abduction:** pick the one pattern that surprised you most. Which theory
   explains it best? What would you need to check in the texts to be sure?

## 4 · Dictionary sentiment: AFINN

The AFINN lexicon gives about 3,300 English words a score from −5 (very
negative) to +5 (very positive). We look up every alphabetic token, then
aggregate per text. Words that are not in the lexicon score 0.

In [ ]:
afinn = Afinn(language="en")

words = df_tokens[df_tokens["token_is_alpha"]].copy()
scores = {w: afinn.score(w) for w in words["token_lower"].unique()}   # look up each word once
words["afinn"] = words["token_lower"].map(scores)

df_afinn = (words.groupby("text_id")
                 .agg(tokens=("afinn", "size"),
                      scored=("afinn", lambda s: (s != 0).sum()),
                      score_sum=("afinn", "sum"),
                      score_mean=("afinn", "mean"))
                 .reindex(TEXT_IDS))
df_afinn["share_scored"] = (df_afinn["scored"] / df_afinn["tokens"]).round(3)
df_afinn["score_mean"] = df_afinn["score_mean"].round(3)
df_afinn.to_csv(OUT_DIR / "sentiment-afinn.tsv", sep="\t")   # overwrite
df_afinn

Which words drive the scores? Check them: is *risk* really negative in a
safety report, and does *not* in front of a word flip its meaning?

In [ ]:
df_afinn_words = (words[words["afinn"] != 0]
                    .groupby(["token_lower", "afinn"]).size()
                    .rename("n").reset_index()
                    .sort_values("n", ascending=False))
df_afinn_words.to_csv(OUT_DIR / "sentiment-afinn-words.tsv", sep="\t", index=False)   # overwrite

pd.concat({"most frequent positive": df_afinn_words[df_afinn_words["afinn"] > 0].head(8).reset_index(drop=True),
           "most frequent negative": df_afinn_words[df_afinn_words["afinn"] < 0].head(8).reset_index(drop=True)},
          axis=1)

In [ ]:
d = df_afinn["score_mean"].rename(index=short)[::-1]

fig, ax = plt.subplots(figsize=(8, 0.45 * len(TEXT_IDS) + 1.5))
ax.barh(d.index, d.values, color=np.where(d.values >= 0, POS_COLOR, NEG_COLOR))
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean AFINN score per token")
ax.set_title(f"AFINN sentiment per text ({ORG})")
fig.tight_layout()
fig.savefig(OUT_DIR / "sentiment-afinn.png", dpi=150)   # overwrite
plt.show()

## 5 · Sentence sentiment: TextBlob

TextBlob scores whole sentences, so it can handle some context, such as
*not* and *very* in front of an adjective. It returns two scores:

- **polarity** from −1 (negative) to +1 (positive)
- **subjectivity** from 0 (factual) to 1 (opinionated)

In [ ]:
df_tb = df_sentences.copy()
blobs = [TextBlob(s).sentiment for s in df_tb["sentence_text"]]
df_tb["polarity"] = [round(b.polarity, 3) for b in blobs]
df_tb["subjectivity"] = [round(b.subjectivity, 3) for b in blobs]
df_tb.to_csv(OUT_DIR / "sentiment-textblob-sentences.tsv", sep="\t", index=False)   # overwrite

df_textblob = (df_tb.groupby("text_id")
                    .agg(sentences=("polarity", "size"),
                         polarity=("polarity", "mean"),
                         subjectivity=("subjectivity", "mean"))
                    .reindex(TEXT_IDS).round(3))
df_textblob.to_csv(OUT_DIR / "sentiment-textblob.tsv", sep="\t")   # overwrite
df_textblob

The most positive and most negative sentences are worth reading: do you agree
with TextBlob?

In [ ]:
cols = ["text_id", "polarity", "sentence_text"]
print("Most positive:")
display(df_tb.nlargest(5, "polarity")[cols])
print("Most negative:")
display(df_tb.nsmallest(5, "polarity")[cols])

In [ ]:
d = df_textblob.rename(index=short)[::-1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 0.45 * len(TEXT_IDS) + 1.5), sharey=True)
ax1.barh(d.index, d["polarity"], color=np.where(d["polarity"] >= 0, POS_COLOR, NEG_COLOR))
ax1.axvline(0, color="black", linewidth=0.8)
ax1.set_xlabel("Mean polarity (−1 to +1)")
ax2.barh(d.index, d["subjectivity"], color=ORANGE)
ax2.set_xlim(0, 1)
ax2.set_xlabel("Mean subjectivity (0 to 1)")
fig.suptitle(f"TextBlob sentiment per text, sentence averages ({ORG})")
fig.tight_layout()
fig.savefig(OUT_DIR / "sentiment-textblob.png", dpi=150)   # overwrite
plt.show()

## 6 · Document-term matrix: frequency and TF-IDF

A **document-term matrix** (DTM) has one row per text and one column per
term, and each cell counts how often the term occurs in that text. We build it
from the kept lemmas, so the corpus vocabulary is the same as in section 2.

**TF-IDF** reweights the counts. Terms that are frequent in one text but rare
in the others get high scores, so TF-IDF shows what makes each text
*different* from the rest.

In [ ]:
docs = kept.groupby("text_id")["token_lemma"].apply(list).reindex(TEXT_IDS)

vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)   # tokens are already lemmas
counts = vectorizer.fit_transform(docs)
terms = vectorizer.get_feature_names_out()

dtm_freq = pd.DataFrame(counts.toarray(), index=TEXT_IDS, columns=terms)
dtm_tfidf = pd.DataFrame(TfidfTransformer().fit_transform(counts).toarray(),
                         index=TEXT_IDS, columns=terms).round(4)

dtm_freq.to_csv(OUT_DIR / f"dtm-freq-{ORG}.tsv", sep="\t", index_label="text_id")    # overwrite
dtm_tfidf.to_csv(OUT_DIR / f"dtm-tfidf-{ORG}.tsv", sep="\t", index_label="text_id")  # overwrite
print(f"DTM: {dtm_freq.shape[0]} texts × {dtm_freq.shape[1]} terms")

The full matrix has far too many columns to plot, so we show two subsets:

- **Frequency:** the 15 most frequent terms in the whole corpus
- **TF-IDF:** the 2 highest-scoring terms of each text

In [ ]:
freq_terms = dtm_freq.sum().sort_values(ascending=False).head(15).index
tfidf_terms = list(dict.fromkeys(t for _, row in dtm_tfidf.iterrows()
                                 for t in row.sort_values(ascending=False).head(2).index))

def heatmap(ax, matrix, fmt, title):
    ax.imshow(matrix.values, cmap="Oranges", aspect="auto")
    ax.set_xticks(range(matrix.shape[1]), matrix.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(matrix.shape[0]), [short(t) for t in matrix.index], fontsize=8)
    top = matrix.values.max()
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix.values[i, j]
            ax.text(j, i, format(v, fmt), ha="center", va="center", fontsize=6,
                    color="white" if v > 0.6 * top else "black")
    ax.set_title(title, fontsize=10)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 0.5 * len(TEXT_IDS) + 2.5))
heatmap(ax1, dtm_freq[freq_terms], "d", "Frequency: 15 most frequent terms")
heatmap(ax2, dtm_tfidf[tfidf_terms], ".2f", "TF-IDF: top 2 terms per text")
ax2.set_yticks([])
fig.suptitle(f"Document-term matrix ({ORG})")
fig.tight_layout()
fig.savefig(OUT_DIR / f"dtm-{ORG}.png", dpi=150)   # overwrite
plt.show()

Compare the two panels. The frequent terms are shared by most texts, while
the TF-IDF terms are each text's **distinctive** vocabulary. Do the
distinctive terms match the page titles? Which ones would you follow up with
KWIC?

## Output files

All files below are in `text-data/lab3/` and are overwritten on every run.

In [ ]:
pd.DataFrame({"file": [p.name for p in sorted(OUT_DIR.iterdir())],
              "size_kb": [round(p.stat().st_size / 1024, 1) for p in sorted(OUT_DIR.iterdir())]})

## Download the results (on Colab)

Files in `/content` are deleted when the Colab session ends. This cell packs
the `text-data/lab3` folder into `lab3-results.zip` and downloads it to your
computer. Locally, the cell does nothing.

In [ ]:
if IN_COLAB:
    from google.colab import files

    zip_path = shutil.make_archive("lab3-results", "zip", root_dir=DATA_DIR, base_dir="lab3")   # overwrite
    files.download(zip_path)